# SAM 3 Image Segmentation with OpenVINO

[SAM 3](https://github.com/facebookresearch/sam3) is Meta's unified vision-language model that supports:
- **Text-prompted segmentation**: Segment objects by describing them in natural language
- **Box-prompted segmentation**: Use bounding boxes as visual prompts (positive/negative)
- **Combined prompts**: Text + negative boxes to exclude regions
- **Point-prompted segmentation** (SAM1 task): Click foreground/background points
- **Video object tracking** (SAM2 task): Track objects across video frames

This notebook demonstrates how to convert SAM 3 models to OpenVINO IR format and run inference using OpenVINO,
aligned with the [official examples on HuggingFace](https://huggingface.co/facebook/sam3).

The model is decomposed into multiple sub-models for efficient conversion:
1. **Image Encoder** — ViT backbone with FPN neck
2. **Text Encoder** — Text transformer for language understanding
3. **Transformer Encoder** — Multi-level feature fusion with text cross-attention
4. **Transformer Decoder** — Object query decoding with box refinement
5. **Scoring** — Dot-product scoring + final box prediction
6. **Segmentation Head** — Pixel decoder + mask prediction
7. **SAM2 Prompt Encoder** — Point/box prompt encoding (for SAM1/Tracker task)
8. **SAM2 Mask Decoder** — Mask prediction from prompts (for SAM1/Tracker task)

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Load PyTorch Model](#Load-PyTorch-Model)
- [PyTorch Baseline Demos](#PyTorch-Baseline-Demos)
  - [Basic Usage](#Basic-Usage)
  - [Text-Only Prompts](#Text-Only-Prompts)
  - [Single Bounding Box Prompt](#Single-Bounding-Box-Prompt)
  - [Multiple Box Prompts (Positive and Negative)](#Multiple-Box-Prompts)
  - [Combined Prompts (Text + Negative Box)](#Combined-Prompts)
  - [SAM3 Tracker](#SAM3-Tracker)
- [Convert Models to OpenVINO](#Convert-Models-to-OpenVINO)
- [OpenVINO Demos](#OpenVINO-Demos)
- [Testing & Verification](#Testing-and-Verification)
- [Performance Comparison](#Performance-Comparison)
- [Interactive Demo](#Interactive-Demo)

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

from pip_helper import pip_install

pip_install(
    "-q",
    "torch>=2.4",
    "torchvision",
    "openvino>=2024.4",
    "Pillow",
    "matplotlib",
    "numpy<2",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
pip_install(
    "-q",
    "timm>=1.0.17",
    "ftfy==6.1.1",
    "regex",
    "iopath>=0.1.10",
    "scikit-image",
    "scikit-learn",
    "pycocotools",
    "gradio>=4.19",
    "modelscope",
    "einops",
    "decord",
    "huggingface_hub",
    "opencv-python-headless",
)

In [ ]:
import subprocess
import sys
import types
from pathlib import Path

# ---------------------------------------------------------------------------
# 1. Clone SAM3 source code
# ---------------------------------------------------------------------------
sam3_root = Path("sam3")
if not sam3_root.exists():
    print("Cloning SAM3 repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/facebookresearch/sam3.git", str(sam3_root)],
        check=True,
    )
    print("SAM3 cloned.")
else:
    print(f"SAM3 source already exists at {sam3_root}")

# ---------------------------------------------------------------------------
# 2. Download SAM3 model weights from ModelScope
# ---------------------------------------------------------------------------
model_dir = Path("sam3_model")
checkpoint = model_dir / "sam3.pt"
if not checkpoint.exists():
    print("Downloading SAM3 weights from ModelScope (facebook/sam3)...")
    try:
        from modelscope import snapshot_download
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelscope"], check=True)
        from modelscope import snapshot_download
    snapshot_download("facebook/sam3", local_dir=str(model_dir))
    print(f"Weights downloaded to {model_dir}")
else:
    print(f"SAM3 checkpoint already exists at {checkpoint}")

# ---------------------------------------------------------------------------
# 3. Patch SAM3 source for CPU-only inference (no CUDA).
# ---------------------------------------------------------------------------
def _patch_file(path, replacements):
    p = Path(path)
    text = p.read_text(encoding="utf-8")
    for old, new in replacements:
        text = text.replace(old, new)
    p.write_text(text, encoding="utf-8")

sam3_pkg = sam3_root / "sam3"

_patch_file(sam3_pkg / "model" / "position_encoding.py", [
    ('device="cuda")', 'device="cpu")'),
])

_patch_file(sam3_pkg / "model" / "decoder.py", [
    ('device_type="cuda", enabled=False', 'device_type="cpu", enabled=False'),
    ('feat_size, feat_size, device="cuda"', 'feat_size, feat_size, device="cpu"'),
])

for rel_path in [
    "model/geometry_encoders.py",
    "model/sam3_tracker_base.py",
    "model/sam3_video_inference.py",
]:
    _patch_file(sam3_pkg / rel_path, [
        (".pin_memory().to(", ".to("),
    ])

_patch_file(sam3_pkg / "model" / "sam3_video_inference.py", [
    ('@torch.autocast(device_type="cuda", dtype=torch.bfloat16)', '# autocast disabled on CPU'),
])

_patch_file(sam3_pkg / "model" / "sam3_tracking_predictor.py", [
    ('torch.autocast(device_type="cuda", dtype=torch.bfloat16)',
     'torch.autocast(device_type="cpu", enabled=False)'),
])

print("CPU patches applied.")

# ---------------------------------------------------------------------------
# 4. Mock triton (CUDA-only, not needed for CPU/OpenVINO)
#    torch._inductor deeply probes triton submodules, so we mock them all.
# ---------------------------------------------------------------------------
if "triton" not in sys.modules:
    class _ConstExpr: pass
    class _DType: pass
    class _AttrsDescriptor: pass

    def _jit(*a, **kw):
        if len(a) == 1 and callable(a[0]): return a[0]
        def w(f): return f
        return w

    def _make(name):
        m = types.ModuleType(name)
        m.__path__ = []
        sys.modules[name] = m
        return m

    triton = _make("triton")
    triton.jit = _jit
    triton.cdiv = lambda a, b: (a + b - 1) // b

    tl = _make("triton.language")
    tl.constexpr = _ConstExpr
    tl.dtype = _DType
    triton.language = tl

    tb = _make("triton.backends")
    triton.backends = tb
    tbc = _make("triton.backends.compiler")
    tbc.AttrsDescriptor = _AttrsDescriptor
    tb.compiler = tbc

    tc = _make("triton.compiler")
    triton.compiler = tc
    tcc = _make("triton.compiler.compiler")
    tc.compiler = tcc

    tr = _make("triton.runtime")
    triton.runtime = tr
    _make("triton.runtime.autotuner")
    tr.autotuner = sys.modules["triton.runtime.autotuner"]
    trj = _make("triton.runtime.jit")
    tr.jit = trj

# Add sam3 package to path
sys.path.insert(0, str(sam3_root))
print("Environment ready.")

In [3]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

import openvino as ov

# Suppress unnecessary warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

## Load PyTorch Model
[back to top ⬆️](#Table-of-contents:)

In [4]:
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

bpe_path = str(sam3_root / "sam3" / "assets" / "bpe_simple_vocab_16e6.txt.gz")
checkpoint_path = str(Path("sam3_model") / "sam3.pt")

model = build_sam3_image_model(
    bpe_path=bpe_path,
    device="cpu",
    eval_mode=True,
    checkpoint_path=checkpoint_path,
    load_from_HF=False,
    enable_inst_interactivity=True,
    compile=False,
)
print(f"Model loaded on CPU with {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

Model loaded on CPU with 860.1M parameters


In [ ]:
# Load test images
image_path = str(sam3_root / "assets" / "images" / "test_image.jpg")
image = Image.open(image_path)
width, height = image.size
print(f"Test image size: {width}x{height}")

# Download COCO images used in the HuggingFace SAM3 examples
coco_dir = Path("test_images")
coco_dir.mkdir(exist_ok=True)

coco_images = {
    "cat_remote.jpg": "http://images.cocodataset.org/val2017/000000077595.jpg",
    "kitchen.jpg": "http://images.cocodataset.org/val2017/000000136466.jpg",
}

for fname, url in coco_images.items():
    fpath = coco_dir / fname
    if not fpath.exists():
        print(f"Downloading {fname}...")
        r = requests.get(url)
        fpath.write_bytes(r.content)
    else:
        print(f"{fname} already downloaded")

# Also check for truck image (used by SAM3 Tracker)
truck_path = sam3_root / "assets" / "images" / "truck.jpg"
if not truck_path.exists():
    truck_url = "https://huggingface.co/datasets/hf-internal-testing/sam2-fixtures/resolve/main/truck.jpg"
    print("Downloading truck.jpg...")
    r = requests.get(truck_url)
    truck_path.write_bytes(r.content)

plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis("off")
plt.title("Test Image")
plt.show()

## PyTorch Baseline Demos
[back to top ⬆️](#Table-of-contents:)

Run the original SAM3 model to establish baselines for comparison with OpenVINO results.
These examples are aligned with the [official HuggingFace model card](https://huggingface.co/facebook/sam3).

### Basic Usage

### Text-Only Prompts

In [ ]:
processor = Sam3Processor(model, device="cpu", confidence_threshold=0.5)

# Use COCO image (aligned with HF example)
coco_image = Image.open("test_images/cat_remote.jpg")
coco_w, coco_h = coco_image.size
print(f"COCO image size: {coco_w}x{coco_h}")

inference_state = processor.set_image(coco_image)

processor.reset_all_prompts(inference_state)
inference_state = processor.set_text_prompt(state=inference_state, prompt="ear")

pt_text_masks = inference_state["masks"].clone()
pt_text_scores = inference_state["scores"].clone()

img0 = Image.open("test_images/cat_remote.jpg")
plot_results(img0, inference_state)
print(f"Found {len(pt_text_masks)} object(s) with scores: {pt_text_scores.tolist()}")

### Single Bounding Box Prompt

In [ ]:
# Box in xyxy pixel format [x1, y1, x2, y2] (matching HF example format)
# Using the test image for this demo
inference_state = processor.set_image(image)
processor.reset_all_prompts(inference_state)

box_xyxy = [480.0, 290.0, 590.0, 650.0]  # right shoe region
# Convert xyxy to normalized cxcywh for SAM3 native API
x1, y1, x2, y2 = box_xyxy
cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
bw, bh = x2 - x1, y2 - y1
norm_box_cxcywh = [cx / width, cy / height, bw / width, bh / height]

inference_state = processor.add_geometric_prompt(
    state=inference_state, box=norm_box_cxcywh, label=True
)

pt_box_masks = inference_state["masks"].clone()

# Visualize input box
img0 = Image.open(image_path)
box_xywh = [box_xyxy[0], box_xyxy[1], box_xyxy[2] - box_xyxy[0], box_xyxy[3] - box_xyxy[1]]
image_with_box = draw_box_on_image(img0, box_xywh)
plt.figure(figsize=(10, 8))
plt.imshow(image_with_box)
plt.axis("off")
plt.title(f"Input Box (xyxy): {box_xyxy}")
plt.show()

img0 = Image.open(image_path)
plot_results(img0, inference_state)

### Multiple Box Prompts (Positive and Negative)

In [ ]:
# Two boxes in xyxy format: positive (right shoe) + negative (left shoe)
inference_state = processor.set_image(image)
processor.reset_all_prompts(inference_state)

boxes_xyxy = [[480.0, 290.0, 590.0, 650.0], [370.0, 280.0, 485.0, 655.0]]
box_labels = [True, False]  # True=positive, False=negative

for box_xyxy_i, label in zip(boxes_xyxy, box_labels):
    x1, y1, x2, y2 = box_xyxy_i
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    bw, bh = x2 - x1, y2 - y1
    norm_cxcywh = [cx / width, cy / height, bw / width, bh / height]
    inference_state = processor.add_geometric_prompt(
        state=inference_state, box=norm_cxcywh, label=label
    )

pt_multibox_masks = inference_state["masks"].clone()

# Visualize boxes
img0 = Image.open(image_path)
for i, box_xyxy_i in enumerate(boxes_xyxy):
    x1, y1, x2, y2 = box_xyxy_i
    color = (0, 255, 0) if box_labels[i] else (255, 0, 0)
    img0 = draw_box_on_image(img0, [x1, y1, x2 - x1, y2 - y1], color)
plt.figure(figsize=(10, 8))
plt.imshow(img0)
plt.axis("off")
plt.title("Green=positive, Red=negative")
plt.show()

img0 = Image.open(image_path)
plot_results(img0, inference_state)

### Combined Prompts (Text + Negative Box)

Segment objects matching a text prompt while excluding specific regions using negative bounding boxes.
This example uses the kitchen image: find "handle" but exclude the oven handle region.

In [ ]:
# Combined Prompts: text "handle" + negative box to exclude oven handle region
kitchen_image = Image.open("test_images/kitchen.jpg")
kitchen_w, kitchen_h = kitchen_image.size
print(f"Kitchen image size: {kitchen_w}x{kitchen_h}")

inference_state_kitchen = processor.set_image(kitchen_image)

# Step 1: Set text prompt "handle"
processor.reset_all_prompts(inference_state_kitchen)
inference_state_kitchen = processor.set_text_prompt(state=inference_state_kitchen, prompt="handle")

print(f"Text-only 'handle': found {len(inference_state_kitchen['masks'])} objects")
img0 = Image.open("test_images/kitchen.jpg")
plot_results(img0, inference_state_kitchen)

# Step 2: Add negative box to exclude oven handle region
oven_handle_box_xyxy = [40.0, 183.0, 318.0, 204.0]
x1, y1, x2, y2 = oven_handle_box_xyxy
cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
bw, bh = x2 - x1, y2 - y1
neg_box_cxcywh = [cx / kitchen_w, cy / kitchen_h, bw / kitchen_w, bh / kitchen_h]

inference_state_kitchen = processor.add_geometric_prompt(
    state=inference_state_kitchen, box=neg_box_cxcywh, label=False  # False = negative
)

pt_combined_masks = inference_state_kitchen["masks"].clone()
pt_combined_scores = inference_state_kitchen["scores"].clone()

print(f"Text + negative box: found {len(pt_combined_masks)} objects")

# Visualize
img0 = Image.open("test_images/kitchen.jpg")
box_xywh = [x1, y1, x2 - x1, y2 - y1]
img0 = draw_box_on_image(img0, box_xywh, (255, 0, 0))
plt.figure(figsize=(10, 8))
plt.imshow(img0)
plt.axis("off")
plt.title("Red=negative box (oven handle excluded)")
plt.show()

img0 = Image.open("test_images/kitchen.jpg")
plot_results(img0, inference_state_kitchen)

### SAM3 Tracker

SAM3 includes a tracker model (SAM2-style) for point/box-based interactive segmentation.
This is equivalent to the SAM3 Tracker example on HuggingFace.

In [ ]:
# SAM3 Tracker: single point click on truck image (matching HF example)
truck_image_path = str(sam3_root / "assets" / "images" / "truck.jpg")
truck_image = Image.open(truck_image_path)
truck_w, truck_h = truck_image.size
print(f"Truck image size: {truck_w}x{truck_h}")

plt.figure(figsize=(10, 8))
plt.imshow(truck_image)
plt.plot(500, 375, 'r*', markersize=15)
plt.axis("off")
plt.title("Input point (500, 375)")
plt.show()

# Use the SAM3 interactive predictor (SAM2-style tracker for single image)
# The tracker shares the main model's backbone — set it before calling set_image
if model.inst_interactive_predictor is not None:
    tracker_predictor = model.inst_interactive_predictor
    tracker_predictor.model.backbone = model.backbone
    tracker_predictor.set_image(truck_image)
    
    input_point = np.array([[500, 375]])
    input_label = np.array([1])  # 1 = foreground
    
    pt_tracker_masks, pt_tracker_scores, pt_tracker_logits = tracker_predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True,
    )
    
    best_idx = np.argmax(pt_tracker_scores)
    print(f"Generated {pt_tracker_masks.shape[0]} masks, best score: {pt_tracker_scores[best_idx]:.4f}")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for i in range(min(3, pt_tracker_masks.shape[0])):
        axes[i].imshow(truck_image)
        mask = pt_tracker_masks[i]
        colored = np.zeros((*mask.shape, 4))
        colored[mask > 0.5] = [0.12, 0.56, 1.0, 0.5]
        axes[i].imshow(colored)
        axes[i].set_title(f"Mask {i} (score={pt_tracker_scores[i]:.3f})")
        axes[i].axis("off")
    plt.suptitle("SAM3 Tracker - PyTorch")
    plt.tight_layout()
    plt.show()
else:
    print("Tracker not available (inst_interactivity not enabled)")

## Convert Models to OpenVINO
[back to top ⬆️](#Table-of-contents:)

We decompose the SAM3 model into separate sub-models for OpenVINO conversion.
Each sub-model is wrapped in a dedicated class that produces traceable forward passes.

In [ ]:
from ov_sam3_helper import (
    patch_vit_rope,
    patch_tracker_rope,
    Sam3ImageEncoderModel,
    Sam3TextEncoderModel,
    Sam3TransformerEncoderModel,
    Sam3TransformerDecoderModel,
    Sam3ScoringModel,
    Sam3GeometryEncoderModel,
    Sam3SegmentationHeadModel,
    Sam3SAM2PromptEncoderModel,
    Sam3SAM2MaskDecoderModel,
    convert_and_save_model,
)

ov_model_dir = Path("ov_models")
ov_model_dir.mkdir(exist_ok=True)

### Image Encoder

The image encoder uses a ViT backbone with Rotary Position Encoding (RoPE).
We must replace the complex tensor-based RoPE with matrix multiplication before conversion.

In [10]:
# Patch RoPE to use matrix multiplication (OpenVINO doesn't support complex tensors)
patch_vit_rope(model.backbone.vision_backbone.trunk)

image_encoder_wrapper = Sam3ImageEncoderModel(
    neck=model.backbone.vision_backbone,
    scalp=model.backbone.scalp,
)

ov_image_encoder_path = ov_model_dir / "image_encoder.xml"
ov_image_encoder_model = convert_and_save_model(
    wrapper_model=image_encoder_wrapper,
    example_input=torch.zeros(1, 3, 1008, 1008),
    save_path=str(ov_image_encoder_path),
    model_name="Image Encoder",
)

  [skip] Image Encoder already exists at ov_models\image_encoder.xml


### Text Encoder

In [11]:
text_encoder_wrapper = Sam3TextEncoderModel(
    text_encoder=model.backbone.language_backbone,
)

ov_text_encoder_path = ov_model_dir / "text_encoder.xml"
ov_text_encoder_model = convert_and_save_model(
    wrapper_model=text_encoder_wrapper,
    example_input=torch.zeros(1, 32, dtype=torch.long),
    save_path=str(ov_text_encoder_path),
    model_name="Text Encoder",
)

  [skip] Text Encoder already exists at ov_models\text_encoder.xml


### Transformer Encoder

The transformer encoder fuses image features with text prompt embeddings through cross-attention.

In [12]:
encoder_wrapper = Sam3TransformerEncoderModel(
    encoder=model.transformer.encoder,
    feat_h=72,
    feat_w=72,
)

# Dummy inputs: seq-first (HW, B, C) where H=W=72
seq_len = 72 * 72  # 5184
dummy_img_feat = torch.randn(seq_len, 1, 256)
dummy_img_pos = torch.randn(seq_len, 1, 256)
# Prompt: (seq_len, B, C) - text prompt features
dummy_prompt = torch.randn(10, 1, 256)
dummy_prompt_mask = torch.zeros(1, 10, dtype=torch.bool)

ov_encoder_path = ov_model_dir / "transformer_encoder.xml"
ov_encoder_model = convert_and_save_model(
    wrapper_model=encoder_wrapper,
    example_input=(dummy_img_feat, dummy_img_pos, dummy_prompt, dummy_prompt_mask),
    save_path=str(ov_encoder_path),
    model_name="Transformer Encoder",
)

  [skip] Transformer Encoder already exists at ov_models\transformer_encoder.xml


### Transformer Decoder

The decoder handles object query decoding with cross-attention to encoder memory and text prompts.

### Scoring

The scoring model computes dot-product scores between object queries and text prompts,
and performs final box refinement via bbox_embed + decoder_norm.

In [ ]:
# --- Transformer Decoder ---
decoder_wrapper = Sam3TransformerDecoderModel(
    decoder=model.transformer.decoder,
)

seq_len = 72 * 72  # 5184
dummy_memory = torch.randn(seq_len, 1, 256)
dummy_pos_embed = torch.randn(seq_len, 1, 256)
dummy_memory_mask = torch.zeros(1)
dummy_level_start = torch.tensor([0], dtype=torch.long)
dummy_spatial = torch.tensor([[72, 72]], dtype=torch.long)
dummy_valid_ratios = torch.ones(1, 1, 2)
dummy_prompt_dec = torch.randn(10, 1, 256)
dummy_prompt_mask_dec = torch.zeros(1, 10, dtype=torch.bool)

# Force re-conversion for the new split models
for name in ["decoder", "scoring"]:
    for ext in [".xml", ".bin"]:
        p = ov_model_dir / f"{name}{ext}"
        if p.exists():
            p.unlink()

ov_decoder_path = ov_model_dir / "decoder.xml"
ov_decoder_model = convert_and_save_model(
    wrapper_model=decoder_wrapper,
    example_input=(
        dummy_memory, dummy_pos_embed, dummy_memory_mask,
        dummy_level_start, dummy_spatial, dummy_valid_ratios,
        dummy_prompt_dec, dummy_prompt_mask_dec,
    ),
    save_path=str(ov_decoder_path),
    model_name="Transformer Decoder",
)

# --- Scoring ---
num_queries = model.transformer.decoder.num_queries
num_layers = len(model.transformer.decoder.layers)

scoring_wrapper = Sam3ScoringModel(
    dot_prod_scoring=model.dot_prod_scoring,
    bbox_embed=model.transformer.decoder.bbox_embed,
    decoder_norm=model.transformer.decoder.norm,
)

# Scoring inputs (batch-first, as output by decoder after transpose)
dummy_hs_bf = torch.randn(num_layers, 1, num_queries, 256)
dummy_ref_bf = torch.randn(num_layers + 1, 1, num_queries, 4)
dummy_prompt_sc = torch.randn(10, 1, 256)
dummy_mask_sc = torch.zeros(1, 10, dtype=torch.bool)

ov_scoring_path = ov_model_dir / "scoring.xml"
ov_scoring_model = convert_and_save_model(
    wrapper_model=scoring_wrapper,
    example_input=(dummy_hs_bf, dummy_ref_bf, dummy_prompt_sc, dummy_mask_sc),
    save_path=str(ov_scoring_path),
    model_name="Scoring",
)

# --- Geometry Encoder ---
MAX_BOXES = 10
geo_wrapper = Sam3GeometryEncoderModel(geometry_encoder=model.geometry_encoder)

dummy_box_emb = torch.zeros(MAX_BOXES, 1, 4)
dummy_box_mask = torch.ones(1, MAX_BOXES, dtype=torch.bool)
dummy_box_labels = torch.ones(MAX_BOXES, 1, dtype=torch.long)
dummy_img_feat = torch.randn(72 * 72, 1, 256)
dummy_img_pos = torch.randn(72 * 72, 1, 256)

ov_geometry_encoder_path = ov_model_dir / "geometry_encoder.xml"
ov_geometry_encoder_model = convert_and_save_model(
    wrapper_model=geo_wrapper,
    example_input=(
        dummy_box_emb, dummy_box_mask, dummy_box_labels,
        dummy_img_feat, dummy_img_pos,
    ),
    save_path=str(ov_geometry_encoder_path),
    model_name="Geometry Encoder",
)

### Segmentation Head

The segmentation head includes a pixel decoder (FPN upsampling), instance segmentation head, and mask predictor.

In [14]:
num_queries = model.transformer.decoder.num_queries
num_layers = len(model.transformer.decoder.layers)

seg_wrapper = Sam3SegmentationHeadModel(
    seg_head=model.segmentation_head,
)

# FPN features at 3 resolution levels (after scalp=1)
seq_len = 72 * 72  # 5184
dummy_fpn0 = torch.randn(1, 256, 288, 288)
dummy_fpn1 = torch.randn(1, 256, 144, 144)
dummy_fpn2 = torch.randn(1, 256, 72, 72)
dummy_obj_queries = torch.randn(1, num_queries, 256)
dummy_enc_states = torch.randn(seq_len, 1, 256)
dummy_prompt_seg = torch.randn(10, 1, 256)
dummy_mask_seg = torch.zeros(1, 10, dtype=torch.bool)

ov_seg_path = ov_model_dir / "segmentation_head.xml"
ov_seg_model = convert_and_save_model(
    wrapper_model=seg_wrapper,
    example_input=(
        dummy_fpn0, dummy_fpn1, dummy_fpn2,
        dummy_obj_queries, dummy_enc_states,
        dummy_prompt_seg, dummy_mask_seg,
    ),
    save_path=str(ov_seg_path),
    model_name="Segmentation Head",
)

  [skip] Segmentation Head already exists at ov_models\segmentation_head.xml


### SAM2-Style Prompt Encoder (for SAM1 Task)

Used for point/box prompting with the interactive predictor.

In [15]:
if model.inst_interactive_predictor is not None:
    tracker_model = model.inst_interactive_predictor.model
    
    prompt_enc_wrapper = Sam3SAM2PromptEncoderModel(
        prompt_encoder=tracker_model.sam_prompt_encoder,
        image_size=tracker_model.image_size,
    )

    dummy_coords = torch.zeros(1, 3, 2)  # (B, N_points, 2)
    dummy_labels = torch.ones(1, 3, dtype=torch.int32)  # (B, N_points)
    dummy_has_box = torch.tensor(0)

    ov_prompt_enc_path = ov_model_dir / "sam2_prompt_encoder.xml"
    ov_prompt_enc_model = convert_and_save_model(
        wrapper_model=prompt_enc_wrapper,
        example_input=(dummy_coords, dummy_labels, dummy_has_box),
        save_path=str(ov_prompt_enc_path),
        model_name="SAM2 Prompt Encoder",
    )
else:
    print("Skipping SAM2 Prompt Encoder (inst_interactivity not enabled)")

  [skip] SAM2 Prompt Encoder already exists at ov_models\sam2_prompt_encoder.xml


### SAM2-Style Mask Decoder (for SAM1 Task)

In [16]:
if model.inst_interactive_predictor is not None:
    mask_dec_wrapper = Sam3SAM2MaskDecoderModel(
        model=tracker_model,
        multimask_output=True,
    )

    # SAM2 mask decoder inputs
    dummy_img_embed = torch.randn(1, 256, 72, 72)
    dummy_hires_0 = torch.randn(1, 32, 288, 288)
    dummy_hires_1 = torch.randn(1, 64, 144, 144)
    dummy_sparse = torch.randn(1, 3, 256)
    dummy_dense = torch.randn(1, 256, 72, 72)

    ov_mask_dec_path = ov_model_dir / "sam2_mask_decoder.xml"
    ov_mask_dec_model = convert_and_save_model(
        wrapper_model=mask_dec_wrapper,
        example_input=(
            dummy_img_embed, dummy_hires_0, dummy_hires_1,
            dummy_sparse, dummy_dense,
        ),
        save_path=str(ov_mask_dec_path),
        model_name="SAM2 Mask Decoder",
    )
else:
    print("Skipping SAM2 Mask Decoder (inst_interactivity not enabled)")

  [skip] SAM2 Mask Decoder already exists at ov_models\sam2_mask_decoder.xml


## OpenVINO Demos
[back to top ⬆️](#Table-of-contents:)

Now run the same examples using OpenVINO inference pipeline.

In [17]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

In [ ]:
core = ov.Core()

# Compile all detector models
ov_image_encoder = core.compile_model(ov_model_dir / "image_encoder.xml", device.value)
ov_text_encoder = core.compile_model(ov_model_dir / "text_encoder.xml", device.value)
ov_geometry_encoder = core.compile_model(ov_model_dir / "geometry_encoder.xml", device.value)
ov_transformer_encoder = core.compile_model(ov_model_dir / "transformer_encoder.xml", device.value)
ov_decoder = core.compile_model(ov_model_dir / "decoder.xml", device.value)
ov_scoring = core.compile_model(ov_model_dir / "scoring.xml", device.value)
ov_seg_head = core.compile_model(ov_model_dir / "segmentation_head.xml", device.value)

# Compile SAM2 models for SAM1-style point/box prompting (Tracker)
ov_prompt_encoder = None
ov_mask_decoder = None
if (ov_model_dir / "sam2_prompt_encoder.xml").exists():
    ov_prompt_encoder = core.compile_model(ov_model_dir / "sam2_prompt_encoder.xml", device.value)
    print("SAM2 Prompt Encoder compiled")
if (ov_model_dir / "sam2_mask_decoder.xml").exists():
    ov_mask_decoder = core.compile_model(ov_model_dir / "sam2_mask_decoder.xml", device.value)
    print("SAM2 Mask Decoder compiled")

print("All models compiled successfully!")

In [ ]:
from ov_sam3_helper import OVSam3Processor, Sam3GeometryEncoderModel
from sam3.model.tokenizer_ve import SimpleTokenizer

# Tokenizer: load from BPE vocab file
bpe_path = str(sam3_root / "sam3" / "assets" / "bpe_simple_vocab_16e6.txt.gz")
tokenizer = SimpleTokenizer(bpe_path=bpe_path)

# PyTorch geometry encoder wrapper (roi_align doesn't convert well to OV IR)
pt_geo_encoder = Sam3GeometryEncoderModel(geometry_encoder=model.geometry_encoder)
pt_geo_encoder.eval()

# Build OV pipeline with separate decoder and scoring models
ov_processor = OVSam3Processor(
    ov_image_encoder=ov_image_encoder,
    ov_text_encoder=ov_text_encoder,
    ov_geometry_encoder=ov_geometry_encoder,
    ov_transformer_encoder=ov_transformer_encoder,
    ov_decoder=ov_decoder,
    ov_scoring=ov_scoring,
    ov_seg_head=ov_seg_head,
    tokenizer=tokenizer,
    auxiliary=model,
    ov_prompt_encoder=ov_prompt_encoder,
    ov_mask_decoder=ov_mask_decoder,
    confidence_threshold=0.5,
    max_boxes=10,
    pt_geometry_encoder=pt_geo_encoder,
)
print("OVSam3Processor ready (SAM1 task:", "enabled" if ov_processor._sam1_ready else "disabled", ")")

### Text-Only Prompts (OpenVINO)

In [ ]:
# Text-Only Prompts: "ear" on COCO image (matching HF example)
ov_state = ov_processor.set_image(coco_image)

ov_processor.reset_all_prompts(ov_state)
ov_state = ov_processor.set_text_prompt(prompt="ear", state=ov_state)

ov_text_masks = ov_state["masks"].clone()
ov_text_scores = ov_state["scores"].clone()

img0 = Image.open("test_images/cat_remote.jpg")
plot_results(img0, ov_state)
print(f"Found {len(ov_text_masks)} object(s) with scores: {ov_text_scores.tolist()}")

### Single Bounding Box Prompt (OpenVINO)

In [ ]:
# Single box prompt (xyxy format, matching HF example)
ov_state = ov_processor.set_image(image)
ov_processor.reset_all_prompts(ov_state)

box_xyxy = [480.0, 290.0, 590.0, 650.0]
x1, y1, x2, y2 = box_xyxy
cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
bw, bh = x2 - x1, y2 - y1
norm_box_cxcywh = [cx / width, cy / height, bw / width, bh / height]

ov_state = ov_processor.add_geometric_prompt(
    state=ov_state, box=norm_box_cxcywh, label=True
)

ov_box_masks = ov_state["masks"].clone()

img0 = Image.open(image_path)
box_xywh = [x1, y1, x2 - x1, y2 - y1]
image_with_box = draw_box_on_image(img0, box_xywh)
plt.figure(figsize=(10, 8))
plt.imshow(image_with_box)
plt.axis("off")
plt.title(f"Input Box (xyxy): {box_xyxy}")
plt.show()

img0 = Image.open(image_path)
plot_results(img0, ov_state)

### Multiple Box Prompts (OpenVINO)

In [ ]:
# Multiple boxes: positive + negative (xyxy format)
ov_state = ov_processor.set_image(image)
ov_processor.reset_all_prompts(ov_state)

boxes_xyxy = [[480.0, 290.0, 590.0, 650.0], [370.0, 280.0, 485.0, 655.0]]
box_labels = [True, False]

for box_xyxy_i, label in zip(boxes_xyxy, box_labels):
    x1, y1, x2, y2 = box_xyxy_i
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    bw, bh = x2 - x1, y2 - y1
    norm_cxcywh = [cx / width, cy / height, bw / width, bh / height]
    ov_state = ov_processor.add_geometric_prompt(
        state=ov_state, box=norm_cxcywh, label=label
    )

ov_multibox_masks = ov_state["masks"].clone()

# Visualize
img0 = Image.open(image_path)
for i, box_xyxy_i in enumerate(boxes_xyxy):
    x1, y1, x2, y2 = box_xyxy_i
    color = (0, 255, 0) if box_labels[i] else (255, 0, 0)
    img0 = draw_box_on_image(img0, [x1, y1, x2 - x1, y2 - y1], color)
plt.figure(figsize=(10, 8))
plt.imshow(img0)
plt.axis("off")
plt.title("Green=positive, Red=negative")
plt.show()

img0 = Image.open(image_path)
plot_results(img0, ov_state)

### Combined Prompts (Text + Negative Box) (OpenVINO)

In [ ]:
# Combined Prompts (OV): text "handle" + negative box to exclude oven handle
ov_state_kitchen = ov_processor.set_image(kitchen_image)

# Step 1: Set text prompt
ov_processor.reset_all_prompts(ov_state_kitchen)
ov_state_kitchen = ov_processor.set_text_prompt(prompt="handle", state=ov_state_kitchen)
print(f"Text-only 'handle': found {len(ov_state_kitchen['masks'])} objects")

# Step 2: Add negative box
oven_handle_box_xyxy = [40.0, 183.0, 318.0, 204.0]
x1, y1, x2, y2 = oven_handle_box_xyxy
cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
bw, bh = x2 - x1, y2 - y1
neg_box_cxcywh = [cx / kitchen_w, cy / kitchen_h, bw / kitchen_w, bh / kitchen_h]

ov_state_kitchen = ov_processor.add_geometric_prompt(
    state=ov_state_kitchen, box=neg_box_cxcywh, label=False
)

ov_combined_masks = ov_state_kitchen["masks"].clone()
ov_combined_scores = ov_state_kitchen["scores"].clone()

print(f"Text + negative box: found {len(ov_combined_masks)} objects")

# Visualize
img0 = Image.open("test_images/kitchen.jpg")
box_xywh = [x1, y1, x2 - x1, y2 - y1]
img0 = draw_box_on_image(img0, box_xywh, (255, 0, 0))
plt.figure(figsize=(10, 8))
plt.imshow(img0)
plt.axis("off")
plt.title("Red=negative box (oven handle excluded)")
plt.show()

img0 = Image.open("test_images/kitchen.jpg")
plot_results(img0, ov_state_kitchen)

### SAM3 Tracker (OpenVINO)

In [ ]:
# SAM3 Tracker (OV): single point click on truck image
ov_truck_state = ov_processor.set_image(truck_image)

input_point = np.array([[500, 375]])
input_label = np.array([1])

ov_tracker_masks, ov_tracker_scores, ov_tracker_logits = ov_processor.predict_inst(
    ov_truck_state,
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

best_idx = np.argmax(ov_tracker_scores.flatten())
print(f"Generated {ov_tracker_masks.shape[0]} masks, best score: {ov_tracker_scores.flatten()[best_idx]:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i in range(min(3, ov_tracker_masks.shape[0])):
    axes[i].imshow(truck_image)
    mask = ov_tracker_masks[i]
    colored = np.zeros((*mask.shape[-2:], 4))
    mask_2d = mask.squeeze()
    colored[mask_2d > 0.5] = [0.12, 0.56, 1.0, 0.5]
    axes[i].imshow(colored)
    score = ov_tracker_scores.flatten()[i]
    axes[i].set_title(f"Mask {i} (score={score:.3f})")
    axes[i].axis("off")
plt.suptitle("SAM3 Tracker - OpenVINO")
plt.tight_layout()
plt.show()

## Testing and Verification
[back to top ⬆️](#Table-of-contents:)

Compare PyTorch and OpenVINO outputs to verify conversion correctness.

In [ ]:
from ov_sam3_helper import compare_masks, cosine_similarity

results = []

# Test 1: Text prompt masks comparison
if pt_text_masks.shape == ov_text_masks.shape:
    iou = compare_masks(pt_text_masks, ov_text_masks, title="Text Prompt: PyTorch vs OpenVINO")
    results.append(("Text prompt 'ear'", f"IoU={iou:.4f}", "PASS" if iou > 0.90 else "FAIL"))
else:
    results.append(("Text prompt 'ear'", f"Shape mismatch: PT={pt_text_masks.shape} vs OV={ov_text_masks.shape}", "FAIL"))

# Test 2: Box prompt masks comparison
if pt_box_masks.shape == ov_box_masks.shape:
    iou = compare_masks(pt_box_masks, ov_box_masks, title="Box Prompt: PyTorch vs OpenVINO")
    results.append(("Single box prompt", f"IoU={iou:.4f}", "PASS" if iou > 0.90 else "FAIL"))
else:
    results.append(("Single box prompt", f"Shape mismatch: PT={pt_box_masks.shape} vs OV={ov_box_masks.shape}", "FAIL"))

# Test 3: Multi-box masks comparison
if pt_multibox_masks.shape == ov_multibox_masks.shape:
    iou = compare_masks(pt_multibox_masks, ov_multibox_masks, title="Multi-box: PyTorch vs OpenVINO")
    results.append(("Multi-box prompt", f"IoU={iou:.4f}", "PASS" if iou > 0.90 else "FAIL"))
else:
    results.append(("Multi-box prompt", f"Shape mismatch: PT={pt_multibox_masks.shape} vs OV={ov_multibox_masks.shape}", "FAIL"))

# Test 4: Combined prompts comparison
if pt_combined_masks.shape == ov_combined_masks.shape:
    iou = compare_masks(pt_combined_masks, ov_combined_masks, title="Combined Prompts: PyTorch vs OpenVINO")
    results.append(("Combined prompts", f"IoU={iou:.4f}", "PASS" if iou > 0.90 else "FAIL"))
else:
    results.append(("Combined prompts", f"Shape mismatch: PT={pt_combined_masks.shape} vs OV={ov_combined_masks.shape}", "FAIL"))

# Test 5: SAM3 Tracker comparison
if model.inst_interactive_predictor is not None:
    pt_best = pt_tracker_masks[np.argmax(pt_tracker_scores)]
    ov_best = ov_tracker_masks[np.argmax(ov_tracker_scores.flatten())]
    pt_t = torch.from_numpy(pt_best).float()
    ov_t = torch.from_numpy(ov_best.squeeze()).float()
    if pt_t.shape == ov_t.shape:
        iou = compare_masks(pt_t, ov_t, title="SAM3 Tracker: PyTorch vs OpenVINO")
        results.append(("SAM3 Tracker", f"IoU={iou:.4f}", "PASS" if iou > 0.90 else "FAIL"))
    else:
        results.append(("SAM3 Tracker", f"Shape mismatch: PT={pt_t.shape} vs OV={ov_t.shape}", "FAIL"))
else:
    results.append(("SAM3 Tracker", "Skipped (no tracker)", "SKIP"))

In [ ]:
# Test 4: Image encoder unit test
print("\n=== Per-Model Unit Tests ===")

with torch.inference_mode():
    # Prepare test input
    from torchvision.transforms import v2
    transform = v2.Compose([
        v2.ToDtype(torch.uint8, scale=True),
        v2.Resize(size=(1008, 1008)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
    img_tensor = v2.functional.to_image(image)
    test_input = transform(img_tensor).unsqueeze(0)

    # PyTorch image encoder
    pt_result = image_encoder_wrapper(test_input)
    # OV image encoder
    ov_result = ov_image_encoder(test_input.numpy())

    for i in range(len(pt_result)):
        pt_feat = pt_result[i].numpy()
        ov_feat = np.array(ov_result[i])
        sim = cosine_similarity(pt_feat, ov_feat)
        name = f"Image Encoder output {i}"
        status = "PASS" if sim > 0.99 else "FAIL"
        results.append((name, f"cosine_sim={sim:.6f}", status))
        print(f"  {name}: cosine_sim={sim:.6f} [{status}]")

In [ ]:
# Print summary
print("\n" + "=" * 60)
print("VERIFICATION SUMMARY")
print("=" * 60)
print(f"{'Test':<35} {'Metric':<25} {'Status'}")
print("-" * 60)
for test_name, metric, status in results:
    marker = "✓" if status == "PASS" else "✗"
    print(f"{marker} {test_name:<33} {metric:<25} {status}")

n_pass = sum(1 for _, _, s in results if s == "PASS")
n_total = len(results)
print(f"\nOverall: {n_pass}/{n_total} tests passed")

## Performance Comparison
[back to top ⬆️](#Table-of-contents:)

Compare inference latency between PyTorch (CPU) and OpenVINO.

In [ ]:
import time

# Warm up
_ = ov_processor.set_image(image)

# Benchmark OV image encoder
n_runs = 3
ov_times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    _ = ov_processor.set_image(image)
    ov_times.append(time.perf_counter() - t0)

# Benchmark PyTorch image encoder
pt_times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    _ = processor.set_image(image)
    pt_times.append(time.perf_counter() - t0)

ov_avg = np.mean(ov_times)
pt_avg = np.mean(pt_times)

print(f"Image Encoding Latency ({n_runs} runs average):")
print(f"  PyTorch (CPU):  {pt_avg:.3f}s")
print(f"  OpenVINO ({device.value}): {ov_avg:.3f}s")
print(f"  Speedup: {pt_avg / ov_avg:.2f}x")

In [ ]:
# Full pipeline benchmark (text prompt)
ov_full_times = []
for _ in range(n_runs):
    t0 = time.perf_counter()
    s = ov_processor.set_image(image)
    ov_processor.reset_all_prompts(s)
    s = ov_processor.set_text_prompt(prompt="shoe", state=s)
    ov_full_times.append(time.perf_counter() - t0)

pt_full_times = []
pt_proc = Sam3Processor(model, device="cpu", confidence_threshold=0.5)
for _ in range(n_runs):
    t0 = time.perf_counter()
    s = pt_proc.set_image(image)
    pt_proc.reset_all_prompts(s)
    s = pt_proc.set_text_prompt(state=s, prompt="shoe")
    pt_full_times.append(time.perf_counter() - t0)

ov_full_avg = np.mean(ov_full_times)
pt_full_avg = np.mean(pt_full_times)

print(f"\nFull Pipeline Latency (text prompt, {n_runs} runs average):")
print(f"  PyTorch (CPU):  {pt_full_avg:.3f}s")
print(f"  OpenVINO ({device.value}): {ov_full_avg:.3f}s")
print(f"  Speedup: {pt_full_avg / ov_full_avg:.2f}x")

In [ ]:
# Model size comparison
ov_model_files = list(ov_model_dir.glob("*.bin"))
total_ov_size = sum(f.stat().st_size for f in ov_model_files)

ckpt_path = Path("sam3_model") / "sam3.pt"
pt_size = ckpt_path.stat().st_size if ckpt_path.exists() else 0

print(f"\nModel Size Comparison:")
print(f"  PyTorch checkpoint: {pt_size / 1e9:.2f} GB")
print(f"  OpenVINO IR total:  {total_ov_size / 1e9:.2f} GB")
print(f"\nIndividual OV model sizes:")
for f in sorted(ov_model_dir.glob("*.bin")):
    print(f"  {f.stem}: {f.stat().st_size / 1e6:.1f} MB")

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Use the interactive Gradio demo below to try SAM3 with your own images. You can:
- Upload an image and enter a text prompt to segment objects
- Draw bounding boxes to specify regions of interest
- Click points for SAM1-style foreground/background prompting

In [ ]:
import gradio as gr
import numpy as np
import torch
from PIL import Image


def segment_text(image, text_prompt, confidence):
    """Segment objects by text prompt."""
    if image is None or not text_prompt.strip():
        return None
    pil_image = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    state = ov_processor.set_image(pil_image)
    ov_processor.confidence_threshold = confidence
    ov_processor.reset_all_prompts(state)
    state = ov_processor.set_text_prompt(prompt=text_prompt, state=state)
    masks = state.get("masks")
    scores = state.get("scores")
    if masks is None or len(masks) == 0:
        return image
    return _overlay_masks(image, masks, scores)


def segment_point(image, point_type, evt: gr.SelectData):
    """Segment by clicking a point on the image."""
    if image is None:
        return image
    x, y = evt.index
    pil_image = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    state = ov_processor.set_image(pil_image)
    label = 1 if point_type == "Foreground" else 0
    masks, scores, _ = ov_processor.predict_inst(
        state,
        point_coords=np.array([[x, y]]),
        point_labels=np.array([label]),
        multimask_output=True,
    )
    best_idx = np.argmax(scores.flatten())
    mask = masks[best_idx]
    return _overlay_single_mask(image, mask)


def segment_box(image, box_input):
    """Segment by bounding box [x0, y0, x1, y1]."""
    if image is None or not box_input.strip():
        return image
    try:
        coords = [float(x.strip()) for x in box_input.split(",")]
        assert len(coords) == 4
    except Exception:
        return image
    pil_image = Image.fromarray(image) if isinstance(image, np.ndarray) else image
    state = ov_processor.set_image(pil_image)
    input_box = np.array(coords)
    masks, scores, _ = ov_processor.predict_inst(
        state,
        box=input_box,
        multimask_output=False,
    )
    return _overlay_single_mask(image, masks[0])


def _overlay_masks(image, masks, scores=None):
    """Overlay multiple masks on image with different colors."""
    import matplotlib.cm as cm
    img = np.array(image).copy() if not isinstance(image, np.ndarray) else image.copy()
    colors = cm.tab10(np.linspace(0, 1, 10))[:, :3]
    for i, mask in enumerate(masks):
        m = mask.cpu().numpy().squeeze() if isinstance(mask, torch.Tensor) else np.squeeze(mask)
        color = (colors[i % len(colors)] * 255).astype(np.uint8)
        img[m > 0.5] = img[m > 0.5] * 0.5 + color * 0.5
    return img


def _overlay_single_mask(image, mask, color=(30, 144, 255)):
    """Overlay a single mask on image."""
    img = np.array(image).copy() if not isinstance(image, np.ndarray) else image.copy()
    m = mask.squeeze()
    if isinstance(m, torch.Tensor):
        m = m.cpu().numpy()
    color = np.array(color, dtype=np.uint8)
    img[m > 0.5] = (img[m > 0.5] * 0.5 + color * 0.5).astype(np.uint8)
    return img


with gr.Blocks(title="SAM3 OpenVINO Demo") as demo:
    gr.Markdown("# SAM 3 Segmentation with OpenVINO")
    with gr.Tabs():
        with gr.TabItem("Text Prompt"):
            with gr.Row():
                with gr.Column():
                    text_image = gr.Image(label="Input Image", type="numpy")
                    text_input = gr.Textbox(label="Text Prompt", placeholder="e.g. shoe, person, car...")
                    confidence = gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Confidence Threshold")
                    text_btn = gr.Button("Segment", variant="primary")
                with gr.Column():
                    text_output = gr.Image(label="Result")
            text_btn.click(segment_text, [text_image, text_input, confidence], text_output)

        with gr.TabItem("Point Prompt (SAM1)"):
            with gr.Row():
                with gr.Column():
                    point_image = gr.Image(label="Click on image to segment", type="numpy")
                    point_type = gr.Radio(["Foreground", "Background"], value="Foreground", label="Point Type")
                with gr.Column():
                    point_output = gr.Image(label="Result")
            point_image.select(segment_point, [point_image, point_type], point_output)

        with gr.TabItem("Box Prompt (SAM1)"):
            with gr.Row():
                with gr.Column():
                    box_image = gr.Image(label="Input Image", type="numpy")
                    box_input = gr.Textbox(label="Box coordinates (x0, y0, x1, y1)", placeholder="480, 290, 590, 650")
                    box_btn = gr.Button("Segment", variant="primary")
                with gr.Column():
                    box_output = gr.Image(label="Result")
            box_btn.click(segment_box, [box_image, box_input], box_output)

# Skip Gradio launch during headless execution (e.g. nbconvert)
import os as _os
if "NBCONVERT" not in _os.environ and "JUPYTER_EXECUTE" not in _os.environ:
    try:
        demo.launch(debug=True)
    except Exception:
        demo.launch(debug=True, share=True)
else:
    print("Skipping Gradio demo in headless mode")